# 10 - Seguridad y auditoria

Objetivo: pensar seguridad en un MCP que toca GitHub real.


## Grafico guia

```mermaid
flowchart LR
    A["Tool request"] --> B["Validar scope"]
    B --> C["Ejecutar accion permitida"]
    C --> D["Audit log"]
    D --> E["Revision docente/operativa"]
```


In [ ]:
from datetime import datetime, UTC
from uuid import uuid4

server_security = {
    "secret": "GITHUB_TOKEN",
    "default_repo_visibility": "private",
    "dangerous_tools": [],
    "audit_log": "data/github_mcp_audit_log.jsonl",
}
server_security


In [ ]:
def redact(payload):
    redacted = dict(payload)
    for key in ["GITHUB_TOKEN", "token", "authorization"]:
        redacted.pop(key, None)
    if "content" in redacted:
        redacted["content_chars"] = len(redacted.pop("content"))
    return redacted

def audit_event(tool, payload, status="success"):
    return {
        "request_id": "req_" + uuid4().hex[:8],
        "timestamp": datetime.now(UTC).isoformat(),
        "tool": tool,
        "status": status,
        "payload": redact(payload),
    }

audit_event("github_upsert_file", {"repo": "demo/aem4", "path": "README.md", "content": "# Demo", "token": "secret"})


In [ ]:
risk_checklist = [
    "No exponer delete repo",
    "Crear repos privados por defecto",
    "No loguear GITHUB_TOKEN",
    "Registrar tool, repo, path y commit_sha",
    "Pedir confirmacion humana en cliente cuando hay side effects",
]
risk_checklist


## Donde se ve despues

En `github_mcp_utils.py`, `append_audit_event()` escribe un JSONL local sin token ni contenido completo de archivos.
